In [1]:
from io import BytesIO
from urllib.request import urlopen
from zipfile import ZipFile

import pandas as pd
from sklearn.model_selection import train_test_split

In [2]:
URL = (
    "https://archive.ics.uci.edu/static/public/228/"
    "sms+spam+collection.zip"
)

In [3]:
with urlopen(URL, timeout=30) as response:
    zip_content = response.read()

with ZipFile(BytesIO(zip_content)) as archive:
    lines = archive.read("SMSSpamCollection").decode("utf-8").splitlines()

rows = [line.split("\t", maxsplit=1) for line in lines if line]

df = pd.DataFrame(rows, columns=["label", "text"])

assert df["label"].isin(["ham", "spam"]).all()
assert df["text"].str.strip().ne("").all()

df.head()

,label,text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [4]:
# Comprobar que un mismo texto no tenga etiquetas contradictorias.
labels_per_text = df.groupby("text")["label"].nunique()
assert labels_per_text.max() == 1, "Hay textos con etiquetas contradictorias"

# Conservar una sola copia de cada mensaje exacto
df = df.drop_duplicates(subset="text").reset_index(drop=True)

X = df["text"]
y = df["label"].map({"ham":0, "spam":1})

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

assert set(X_train).isdisjoint(set(X_test)), "Hay mensajes duplicados entre train y test"

print("Mensajes en Train:", len(X_train))
print("Mensajes reservados en Test:", len(X_test))

Mensajes en Train: 4136
Mensajes reservados en Test: 1035


In [5]:
class_counts = y_train.value_counts().reindex([0,1], fill_value=0)

class_summary = pd.DataFrame({
    "clase": ["ham (legítimo)", "spam"],
    "cantidad": class_counts.to_numpy(),
    "porcentaje":(
        class_counts / len(y_train) * 100
    ).to_numpy()
})

print(class_summary.round(2).to_string(index=False))

         clase  cantidad  porcentaje
ham (legítimo)      3614       87.38
          spam       522       12.62


In [6]:
majority_accuracy = class_counts.max() / len(y_train)

print(f"Accuracy de la regla mayoritaria en Train: {majority_accuracy:.3f}")

Accuracy de la regla mayoritaria en Train: 0.874


In [7]:
import numpy as np

from sklearn.dummy import DummyClassifier
from sklearn.metrics import make_scorer, precision_score, f1_score
from sklearn.model_selection import StratifiedKFold, cross_validate

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scoring = {
    "accuracy": "accuracy",
    "precision_spam": make_scorer(
        precision_score,
        zero_division=0
    ),
    "recall_spam":"recall",
    "f1_spam": make_scorer(
        f1_score,
        zero_division=0
    )
}
# DummyClassifier no necesita las características del texto
X_dummy = np.zeros((len(y_train), 1))

baseline = DummyClassifier(strategy="most_frequent")

cv_results = cross_validate(
    baseline,
    X_dummy,
    y_train,
    cv=cv,
    scoring=scoring
)

baseline_summary = pd.DataFrame({
    metric: {
        "media": cv_results[f"test_{metric}"].mean(),
        "desviacion": cv_results[f"test_{metric}"].std()
    }
    for metric in scoring
}).T

print(baseline_summary.round(4))

                 media  desviacion
accuracy        0.8738      0.0006
precision_spam  0.0000      0.0000
recall_spam     0.0000      0.0000
f1_spam         0.0000      0.0000


In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer

example_messages = [
    "premio gratis",
    "premio hoy",
    "reunion hoy",
]

example_vectorizer = TfidfVectorizer()

example_matrix = example_vectorizer.fit_transform(example_messages)

example_table = pd.DataFrame(
    example_matrix.toarray(),
    columns=example_vectorizer.get_feature_names_out(),
    index=example_messages
)

print(example_table.round(3))

               gratis    hoy  premio  reunion
premio gratis   0.796  0.000   0.605    0.000
premio hoy      0.000  0.707   0.707    0.000
reunion hoy     0.000  0.605   0.000    0.796


In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

model = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("classifier", LogisticRegression(C=1.0, max_iter=1000))
])

In [10]:
model_cv_results = cross_validate(
    model,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring
)

model_summary = pd.DataFrame({
    metric: {
        "media": model_cv_results[f"test_{metric}"].mean(),
        "desviacion": model_cv_results[f"test_{metric}"].std()
    }
    for metric in scoring
}).T

comparison = pd.DataFrame({
    "baseline": baseline_summary["media"],
    "regresion_logistica": model_summary["media"]
})

print(comparison.round(4))

                baseline  regresion_logistica
accuracy          0.8738               0.9647
precision_spam    0.0000               0.9950
recall_spam       0.0000               0.7240
f1_spam           0.0000               0.8370


In [11]:
from  sklearn.base import clone

c_values = [0.01, 0.1, 1.0, 10.0, 100.0]
results = []

for c in c_values:
    candidate = clone(model)
    candidate.set_params(classifier__C=c)
    
    scores = cross_validate(
        candidate,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring
    )
    
    results.append({
        "C": c,
        **{
            metric: scores[f"test_{metric}"].mean()
            for metric in scoring
        }
    })
    
c_comparison = pd.DataFrame(results).set_index("C")
print(c_comparison.round(4))

        accuracy  precision_spam  recall_spam  f1_spam
C                                                     
0.01      0.8738          0.0000       0.0000   0.0000
0.10      0.8738          0.0000       0.0000   0.0000
1.00      0.9647          0.9950       0.7240   0.8370
10.00     0.9819          0.9809       0.8734   0.9235
100.00    0.9850          0.9735       0.9060   0.9382


In [12]:
from sklearn.model_selection import cross_val_predict

selected_model = clone(model)
selected_model.set_params(classifier__C=10.0)

oof_probabilities = cross_val_predict(
    selected_model,
    X_train,
    y_train,
    cv=cv,
    method="predict_proba"
)

# Las columnas corresponden a las clases 0 y 1
# Seleccionamos la probabilidad de spam: clase 1

oof_spam_probability = oof_probabilities[:,1]


print("Forma de la matriz:", oof_probabilities.shape)
print("Probabilidades de spam:", oof_spam_probability.shape)

Forma de la matriz: (4136, 2)
Probabilidades de spam: (4136,)


In [13]:
oof_preview = pd.DataFrame({
    "etiqueta_real": y_train.to_numpy(),
    "probabilidad_spam": oof_spam_probability
})

print(oof_preview.head().round(4))

   etiqueta_real  probabilidad_spam
0              0             0.0125
1              0             0.0025
2              0             0.0043
3              0             0.0145
4              0             0.0750


In [14]:
from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score
)

thresholds = [0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80]
threshold_results = []

for threshold in thresholds:
    predictions = (oof_spam_probability >= threshold).astype(int)
    
    tn, fp, fn, tp = confusion_matrix(
        y_train,
        predictions,
        labels=[0,1]
    ).ravel()
    
    threshold_results.append({
        "umbral": threshold,
        "precision_spam": precision_score(
            y_train, predictions, zero_division=0
        ),
        "recall_spam": recall_score(
            y_train, predictions, zero_division=0
        ),
        "f1_spam": f1_score(
            y_train, predictions, zero_division=0
        ),
        "FP_legitimos_ocultados": fp,
        "FN_spam_no_detectado": fn,
    })

threshold_table = pd.DataFrame(threshold_results)

print(threshold_table.round(4).to_string(index=False))

 umbral  precision_spam  recall_spam  f1_spam  FP_legitimos_ocultados  FN_spam_no_detectado
    0.2          0.9213       0.9425   0.9318                      42                    30
    0.3          0.9584       0.9272   0.9426                      21                    38
    0.4          0.9752       0.9042   0.9384                      12                    50
    0.5          0.9806       0.8736   0.9240                       9                    66
    0.6          0.9866       0.8448   0.9102                       6                    81
    0.7          0.9906       0.8103   0.8915                       4                    99
    0.8          0.9949       0.7490   0.8546                       2                   131


In [15]:
selected_threshold = 0.50

oof_predictions = (
    oof_spam_probability >= selected_threshold
).astype(int)

error_analysis = pd.DataFrame({
    "texto": X_train.to_numpy(),
    "etiqueta_real": y_train.to_numpy(),
    "probabilidad_spam": oof_spam_probability,
    "prediccion": oof_predictions
})

# Legítimos clasificados incorrectamente como spam.
false_positives = error_analysis[
    (error_analysis["etiqueta_real"] == 0)
    & (error_analysis["prediccion"] == 1)
].sort_values("probabilidad_spam", ascending=False)

# Spam clasificado incorrectamente como legítimo.
false_negatives = error_analysis[
    (error_analysis["etiqueta_real"] == 1)
    & (error_analysis["prediccion"] == 0)
].sort_values("probabilidad_spam")

with pd.option_context("display.max_colwidth", None):
    print("FALSOS POSITIVOS: legítimos ocultados")
    display(false_positives.head(5))
    
    print("FALSOS NEGATIVOS: spam no detectado")
    display(false_negatives.head(5))

FALSOS POSITIVOS: legítimos ocultados


,texto,etiqueta_real,probabilidad_spam,prediccion
1436,Waiting for your call.,0,0.909104,1
1528,K k:) sms chat with me.,0,0.806377,1
2587,K..u also dont msg or reply to his msg..,0,0.739151,1
590,"Hi, Mobile no. &lt;#&gt; has added you in their contact list on www.fullonsms.com It s a great place to send free sms to people For more visit fullonsms.com",0,0.703406,1
3299,I liked the new mobile,0,0.695593,1


FALSOS NEGATIVOS: spam no detectado


,texto,etiqueta_real,probabilidad_spam,prediccion
199,"Do you ever notice that when you're driving, anyone going slower than you is an idiot and everyone driving faster than you is a maniac?",1,0.005116,0
2244,"Hello darling how are you today? I would love to have a chat, why dont you tell me what you look like and what you are in to sexy?",1,0.006622,0
4123,"Hi ya babe x u 4goten bout me?' scammers getting smart..Though this is a regular vodafone no, if you respond you get further prem rate msg/subscription. Other nos used also. Beware!",1,0.010353,0
1786,Hello. We need some posh birds and chaps to user trial prods for champneys. Can i put you down? I need your address and dob asap. Ta r,1,0.011671,0
293,"Do you realize that in about 40 years, we'll have thousands of old ladies running around with tattoos?",1,0.013045,0


In [16]:
final_model = clone(selected_model)
final_model.fit(X_train, y_train)

,steps,"[('tfidf', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,input,'content'
,encoding,'utf-8'
,decode_error,'strict'
,strip_accents,None
,lowercase,True
,preprocessor,None
,tokenizer,None


In [19]:
vectorizer = final_model.named_steps["tfidf"]
classifier = final_model.named_steps["classifier"]

coefficient_table = pd.DataFrame({
    "termino": vectorizer.get_feature_names_out(),
    "coeficiente": classifier.coef_[0]
})

print("Términos con coeficientes más positivos:")
display(
    coefficient_table.nlargest(10, "coeficiente")
    .reset_index(drop=True)
)

print("Términos con coeficientes más negativos:")
display(
    coefficient_table.nsmallest(10, "coeficiente")
    .reset_index(drop=True)
)

print(f"Intercepto: {classifier.intercept_[0]:.4f}")

Términos con coeficientes más positivos:


,termino,coeficiente
0,txt,8.443404
1,text,6.782197
2,reply,6.271900
3,call,5.954791
4,uk,5.935990
5,stop,5.512377
6,www,5.478685
7,free,5.372037
8,50,5.345946
9,150p,5.304066


Términos con coeficientes más negativos:


,termino,coeficiente
0,me,-4.407623
1,my,-4.264263
2,gt,-3.363946
3,lt,-3.337801
4,can,-3.262739
5,that,-3.188822
6,ok,-2.936928
7,at,-2.824240
8,da,-2.742174
9,but,-2.647672


Intercepto: -3.6678


In [20]:
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    log_loss,
    roc_auc_score
)

# Configuración fijada antes de consultar Test.
final_threshold = 0.50

test_probabilities = final_model.predict_proba(X_test)

spam_column = np.flatnonzero(final_model.classes_ == 1)[0]
test_spam_probability = test_probabilities[:, spam_column]

test_predictions = (
    test_spam_probability >= final_threshold
).astype(int)

test_metrics = pd.Series({
    "accuracy": accuracy_score(y_test, test_predictions),
    "precision_spam": precision_score(
        y_test, test_predictions, zero_division=0
    ),
    "recall_spam": recall_score(
        y_test, test_predictions, zero_division=0
    ),
    "f1_spam": f1_score(
        y_test, test_predictions, zero_division=0
    ),
    "roc_auc": roc_auc_score(y_test, test_spam_probability),
    "average_precision": average_precision_score(
        y_test, test_spam_probability
    ),
    "log_loss": log_loss(
        y_test,
        test_probabilities,
        labels=final_model.classes_,
    ),
})

print(test_metrics.round(4))
print(
    "\n¿Cumple Precision ≥ 98 %?:",
    test_metrics["precision_spam"] >= 0.98,
)

accuracy             0.9787
precision_spam       0.9739
recall_spam          0.8550
f1_spam              0.9106
roc_auc              0.9900
average_precision    0.9673
log_loss             0.0700
dtype: float64

¿Cumple Precision ≥ 98 %?: False


In [21]:
test_confusion = confusion_matrix(
    y_test,
    test_predictions,
    labels=[0, 1],
)

test_confusion_table = pd.DataFrame(
    test_confusion,
    index=["Real: legítimo", "Real: spam"],
    columns=["Predicho: legítimo", "Predicho: spam"],
)

display(test_confusion_table)

tn, fp, fn, tp = test_confusion.ravel()

print(f"Legítimos ocultados: {fp}")
print(f"Spam no detectado: {fn}")
print(f"Spam detectado: {tp}")

,Predicho: legítimo,Predicho: spam
Real: legítimo,901,3
Real: spam,19,112


Legítimos ocultados: 3
Spam no detectado: 19
Spam detectado: 112
